# Week 10 — Neural Network Surrogate Training

Same architecture family as W4-W6: MLP with H ∈ {16, 32}, four regularisation variants (plain / dropout / weight-decay / ensemble), 5-fold CV across 8 configs.

Re-trains on the latest data including W6 portal returns (16/16/21/36/26/26/36/46 pts).

F2/F4/F5/F6/F8 had new bests in W6 — fresh NN gradients at these new best points.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_10')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 19 pts, 2D, baseline RMSE = 0.0016
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.0024 ← BEST
    H= 32  dropout     CV RMSE = 0.0028
    H= 16  wd          CV RMSE = 0.0028
    H= 32  ensemble    CV RMSE = 0.0030
    H= 16  plain       CV RMSE = 0.0030
    H= 16  ensemble    CV RMSE = 0.0031
    H= 32  wd          CV RMSE = 0.0036
    H= 32  plain       CV RMSE = 0.0037
  → best: H=16, dropout, RMSE=0.0024 (-44.1% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.01600000075995922, -0.004999999888241291]


F2: 19 pts, 2D, baseline RMSE = 0.2418
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.1979 ← BEST
    H= 32  dropout     CV RMSE = 0.2131
    H= 16  ensemble    CV RMSE = 0.2380
    H= 16  wd          CV RMSE = 0.2450
    H= 16  plain       CV RMSE = 0.2526
    H= 32  wd          CV RMSE = 0.2605
    H= 32  ensemble    CV RMSE = 0.2630
    H= 32  plain       CV RMSE = 0.2728
  → best: H=16, dropout, RMSE=0.1979 (+18.2% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.9890000224113464, 1.1640000343322754]



F3: 24 pts, 3D, baseline RMSE = 0.0727
  All configs (sorted):
    H= 32  wd          CV RMSE = 0.0789 ← BEST
    H= 16  ensemble    CV RMSE = 0.0802
    H= 32  ensemble    CV RMSE = 0.0806
    H= 32  dropout     CV RMSE = 0.0847
    H= 32  plain       CV RMSE = 0.0871
    H= 16  dropout     CV RMSE = 0.0884
    H= 16  wd          CV RMSE = 0.0940
    H= 16  plain       CV RMSE = 0.0981
  → best: H=32, wd, RMSE=0.0789 (-8.6% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.09600000083446503, -0.032999999821186066, -0.7260000109672546]



F4: 39 pts, 4D, baseline RMSE = 9.5753
  All configs (sorted):
    H= 32  wd          CV RMSE = 4.7226 ← BEST
    H= 32  plain       CV RMSE = 4.7473
    H= 16  ensemble    CV RMSE = 4.8182
    H= 32  ensemble    CV RMSE = 4.9962
    H= 16  plain       CV RMSE = 5.1977
    H= 16  wd          CV RMSE = 5.2023
    H= 32  dropout     CV RMSE = 5.4994
    H= 16  dropout     CV RMSE = 5.9236
  → best: H=32, wd, RMSE=4.7226 (+50.7% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-10.435999870300293, 7.335999965667725, -2.5209999084472656, 5.828999996185303]



...

  Gradient dY/dx at best point: [2457.508056640625, 3784.85400390625, 3781.719970703125, 3515.64208984375]



F6: 29 pts, 5D, baseline RMSE = 0.6649
  All configs (sorted):
    H= 16  plain       CV RMSE = 0.3279 ← BEST
    H= 16  wd          CV RMSE = 0.3283
    H= 16  ensemble    CV RMSE = 0.3312
    H= 16  dropout     CV RMSE = 0.3342
    H= 32  ensemble    CV RMSE = 0.3376
    H= 32  wd          CV RMSE = 0.3644
    H= 32  plain       CV RMSE = 0.3760
    H= 32  dropout     CV RMSE = 0.3877
  → best: H=16, plain, RMSE=0.3279 (+50.7% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.7039999961853027, -0.3569999933242798, -1.0160000324249268, -1.2000000476837158, -2.015000104904175]



F7: 39 pts, 6D, baseline RMSE = 0.6437
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3702 ← BEST
    H= 32  ensemble    CV RMSE = 0.3991
    H= 16  dropout     CV RMSE = 0.4242
    H= 32  dropout     CV RMSE = 0.4344
    H= 32  wd          CV RMSE = 0.4678
    H= 32  plain       CV RMSE = 0.4860
    H= 16  wd          CV RMSE = 0.5214
    H= 16  plain       CV RMSE = 0.5481
  → best: H=16, ensemble, RMSE=0.3702 (+42.5% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-1.840000033378601, -3.2780001163482666, 1.621000051498413, 0.4519999921321869, -3.2109999656677246, 0.597000002861023]



F8: 49 pts, 8D, baseline RMSE = 1.1646
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.4152 ← BEST
    H= 32  dropout     CV RMSE = 0.4529
    H= 32  ensemble    CV RMSE = 0.4653
    H= 16  ensemble    CV RMSE = 0.4927
    H= 32  wd          CV RMSE = 0.5010
    H= 32  plain       CV RMSE = 0.5086
    H= 16  wd          CV RMSE = 0.5610
    H= 16  plain       CV RMSE = 0.5705
  → best: H=16, dropout, RMSE=0.4152 (+64.3% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.5509999990463257, -0.2329999953508377, -0.7789999842643738, -0.1679999977350235, 0.03799999877810478, -0.0020000000949949026, -0.5019999742507935, 0.0989999994635582]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 19 pts, 13 positive, 6 zero-or-negative (68% positive)


Sign classifier trained. LOO accuracy = 78.95%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  dropout         0.0024      0.0016    -44.1%       ✗
 2   16  dropout         0.1979      0.2418    +18.2%       ✓
 3   32  wd              0.0789      0.0727     -8.6%       ✗
 4   32  wd              4.7226      9.5753    +50.7%       ✓
 5   32  ensemble       84.7558   1433.8261    +94.1%       ✓
 6   16  plain           0.3279      0.6649    +50.7%       ✓
 7   16  ensemble        0.3702      0.6437    +42.5%       ✓
 8   16  dropout         0.4152      1.1646    +64.3%       ✓
